In [81]:
!pip install pyspark
import os
from google.colab import files
from pyspark.sql import SparkSession
from pyspark.sql.functions import (col, when, avg, max, min, date_format, round, hour, to_date)
from pyspark.sql.types import (StructType, StructField, IntegerType, StringType, DoubleType, TimestampType)
from pathlib import Path


ROOT_DIR = os.getcwd()

csv_path = os.path.join(ROOT_DIR, 'cleaned_market_data.csv')

if not os.path.isfile(csv_path):
  uploaded = files.upload()

assert os.path.isfile(csv_path)

spark = (SparkSession.builder
          .appName("OwlAnalytics")
          .master("local[*]")
          .getOrCreate())

spark_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(csv_path)
)

spark_df = (
    spark_df
    .withColumn("open_time", col("open_time").cast("timestamp"))
    .withColumn("close_time", col("close_time").cast("timestamp"))
)

print(f"Loaded file: {csv_path}")
print(f"Row count: {spark_df.count()}")
print(f"Columns: {", ".join(spark_df.columns)}")
spark_df.printSchema()


Loaded file: /content/cleaned_market_data.csv
Row count: 8559
Columns: symbol, interval, open_time, open, high, low, close, volume, close_time, quote_volume, trade_count, taker_buy_base_volume, taker_buy_quote_volume, price_range, price_change, percent_change, candle_direction
root
 |-- symbol: string (nullable = true)
 |-- interval: string (nullable = true)
 |-- open_time: timestamp (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: double (nullable = true)
 |-- close_time: timestamp (nullable = true)
 |-- quote_volume: double (nullable = true)
 |-- trade_count: double (nullable = true)
 |-- taker_buy_base_volume: string (nullable = true)
 |-- taker_buy_quote_volume: string (nullable = true)
 |-- price_range: double (nullable = true)
 |-- price_change: double (nullable = true)
 |-- percent_change: double (nullable = true)
 |-- candle_direction: string (nullable =

In [82]:
spark_df.createOrReplaceTempView("market_data")

test_rows = spark.sql("""
  SELECT * FROM market_data
  LIMIT 10;
""").count()

print(f"Temporary SQL view created: market_data")
print(f"Test query returned rows: {test_rows}")

Temporary SQL view created: market_data
Test query returned rows: 10


In [83]:
new_df = (
    spark_df
    .withColumn("price_range", col("high") - col("low"))
    .withColumn("price_change", col("close") - col("open"))
    .withColumn(
        "percent_change",
        when(
            col("open") != 0,
            ((col("close") - col("open")) / col("open")) * 100
        )
    )
    .withColumn(
        "candle_direction",
        when(col("close") > col("open"), "up")
        .when(col("close") < col("open"), "down")
        .otherwise("flat")
    )
)

print(f"Created/verified columns:")
created_columns = [c for c in new_df.columns if c not in spark_df.columns]

print(", ".join((created_columns)))
print()

print(f"Example row:")
new_df.select(
    "symbol",
    "price_range",
    "price_change",
    "percent_change",
    "candle_direction"
).show(1, truncate=False)

Created/verified columns:


Example row:
+-------+------------------+----------------+-------------------+----------------+
|symbol |price_range       |price_change    |percent_change     |candle_direction|
+-------+------------------+----------------+-------------------+----------------+
|BNBUSDT|1.6299999999999955|0.67999999999995|0.12204095550888384|up              |
+-------+------------------+----------------+-------------------+----------------+
only showing top 1 row


In [84]:
adj_time_df = (
    new_df.withColumn("trade_date", to_date(col("open_time")))
          .withColumn("trade_hour", hour(col("open_time")))
          .withColumn("day_of_week", date_format(col("open_time"), "E"))
)

adj_time_df.createOrReplaceTempView("market_data")

created_time_columns = [c for c in adj_time_df.columns if c not in new_df.columns]

print(f"Created time features:")
print(", ".join((created_time_columns)))
print()
print(f"Example row:")
adj_time_df.select(
    "symbol",
    "open_time",
    "trade_date",
    "trade_hour",
    "day_of_week"
).show(1, truncate=False)
print()

avg_close_symb = adj_time_df.groupBy("symbol").agg(round(avg(col("close")), 4).alias("average_close"))
print("Average close by symbol:")
avg_close_symb.show()
print()

avg_vol_symb = adj_time_df.groupBy("symbol").agg(round(avg(col("volume")), 2).alias("average_volume"))
print("Average volume by symbol:")
avg_vol_symb.show()
print()

print("Row count by symbol:")
n_row_symb = adj_time_df.groupBy("symbol").count()
n_row_symb.show()
print()

print(
    "Spark analysed every cleaned record whereas the pandas stage analysed only a 50-row sample. "
    "The Spark averages and counts therefore better represent the entire dataset."
)
print()



Created time features:
trade_date, trade_hour, day_of_week

Example row:
+-------+-------------------+----------+----------+-----------+
|symbol |open_time          |trade_date|trade_hour|day_of_week|
+-------+-------------------+----------+----------+-----------+
|BNBUSDT|2026-06-28 01:00:00|2026-06-28|1         |Sun        |
+-------+-------------------+----------+----------+-----------+
only showing top 1 row

Average close by symbol:
+--------+-------------+
|  symbol|average_close|
+--------+-------------+
| BTCUSDT|   64981.5642|
| BNBUSDT|     602.9687|
|DOGEUSDT|       0.0852|
| DOTUSDT|       0.9932|
|LINKUSDT|       8.0594|
| SOLUSDT|      73.4252|
| ADAUSDT|        0.179|
| XRPUSDT|        1.168|
|AVAXUSDT|       7.1661|
| ETHUSDT|    1755.7717|
+--------+-------------+


Average volume by symbol:
+--------+--------------+
|  symbol|average_volume|
+--------+--------------+
| BTCUSDT|         817.6|
| BNBUSDT|       7573.11|
|DOGEUSDT| 2.660615106E7|
| DOTUSDT|     274403.83

In [85]:
from pyspark.sql.functions import dense_rank, count, stddev, sum
from pyspark.sql.window import Window

pre_vol_df = (
    adj_time_df.groupBy("symbol").agg(round(avg("price_range"), 2).alias("avg_price_range"),
                                       round(min("price_range"), 2).alias("min_price_range"),
                                       round(max("price_range"), 2).alias("max_price_range"),
                                       round(stddev("price_range"), 2).alias("stddev_price_range")
                                      )
              )

volatility_window = Window.orderBy(col("stddev_price_range").desc())

vol_df = (
    pre_vol_df.withColumn("rank", dense_rank().over(volatility_window))
    )

print("Volatility ranking:")
vol_df.select("rank", "symbol", "avg_price_range", "stddev_price_range").show()



Volatility ranking:
+----+--------+---------------+------------------+
|rank|  symbol|avg_price_range|stddev_price_range|
+----+--------+---------------+------------------+
|   1| BTCUSDT|         449.01|            320.79|
|   2| ETHUSDT|          15.32|             10.92|
|   3| BNBUSDT|           4.64|              3.51|
|   4| SOLUSDT|           0.81|              0.51|
|   5|LINKUSDT|           0.08|              0.05|
|   5|AVAXUSDT|           0.08|              0.05|
|   6| DOTUSDT|           0.01|              0.01|
|   6| XRPUSDT|           0.01|              0.01|
|   7|DOGEUSDT|            0.0|               0.0|
|   7| ADAUSDT|            0.0|               0.0|
+----+--------+---------------+------------------+



In [86]:
activity_df = spark.sql("""
SELECT
    DENSE_RANK() OVER (
        ORDER BY SUM(trade_count) DESC, SUM(quote_volume) DESC
    ) AS rank,
    symbol,
    SUM(trade_count) AS total_trades,
    SUM(quote_volume) AS total_quote_volume,
    ROUND(AVG(volume), 2) AS average_volume
FROM market_data
GROUP BY symbol
ORDER BY rank
""")

activity_df.show()

+----+--------+------------+--------------------+--------------+
|rank|  symbol|total_trades|  total_quote_volume|average_volume|
+----+--------+------------+--------------------+--------------+
|   1| BTCUSDT|1.35409828E8|4.397679806930999E10|         817.6|
|   2| ETHUSDT| 1.2605759E8|2.012698346490999E10|      13575.28|
|   3| SOLUSDT| 3.4958903E7| 7.038746214290002E9|     114403.58|
|   4| BNBUSDT| 3.2924776E7| 4.066720969350006E9|       7573.11|
|   5| XRPUSDT| 2.5670154E7| 4.070589470739998E9|    4135455.18|
|   6|DOGEUSDT|  2.241964E7|     1.97352155595E9| 2.660615106E7|
|   7|AVAXUSDT|   8382100.0| 6.546937615100006E8|     112251.22|
|   8|LINKUSDT|   5376873.0| 6.757357568799992E8|      98068.73|
|   9| ADAUSDT|   4681223.0| 1.127351952800001E9|    7452852.24|
|  10| DOTUSDT|   1337734.0|2.4212471884999996E8|     274403.83|
+----+--------+------------+--------------------+--------------+



In [88]:
busiest_hour = spark.sql("""
SELECT
    trade_hour,
    SUM(trade_count) AS total_trades
FROM market_data
GROUP BY trade_hour
ORDER BY total_trades DESC
LIMIT 1
""")
print("Busiest hour by trades:")
busiest_hour.show()
print()

busiest_date = spark.sql("""
SELECT
    trade_date,
    SUM(quote_volume) AS total_quote_volume
FROM market_data
GROUP BY trade_date
ORDER BY total_quote_volume DESC
LIMIT 1
""")
print("Busiest date by quote volume:")
busiest_date.show()

Busiest hour by trades:
+----------+------------+
|trade_hour|total_trades|
+----------+------------+
|        15| 3.0928384E7|
+----------+------------+


Busiest date by quote volume:
+----------+-------------------+
|trade_date| total_quote_volume|
+----------+-------------------+
|2026-06-05|5.655621076000001E9|
+----------+-------------------+



In [91]:
volatility_df = spark.sql("""
SELECT
    symbol,
    AVG(price_range) AS avg_price_range,
    MIN(price_range) AS min_price_range,
    MAX(price_range) AS max_price_range,
    STDDEV(price_range) AS stddev_price_range,
    DENSE_RANK() OVER(
        ORDER BY STDDEV(price_range) DESC
    ) AS volatility_rank
FROM market_data
GROUP BY symbol
ORDER BY volatility_rank
""")

activity_df = spark.sql("""
SELECT
    symbol,
    DENSE_RANK() OVER(
        ORDER BY SUM(trade_count) DESC
    ) AS activity_rank
FROM market_data
GROUP BY symbol
ORDER BY activity_rank
""")

summary_df = spark.sql("""
SELECT
    symbol,
    COUNT(*) AS total_records,
    AVG(volume) AS average_volume,
    SUM(trade_count) AS total_trades,
    AVG(percent_change) AS average_percent_change,
    AVG(price_range) AS average_price_range,

    SUM(
        CASE WHEN candle_direction = 'up' THEN 1 ELSE 0 END
    ) AS up_count,

    SUM(
        CASE WHEN candle_direction = 'down' THEN 1 ELSE 0 END
    ) AS down_count,

    SUM(
        CASE WHEN candle_direction = 'flat' THEN 1 ELSE 0 END
    ) AS flat_count

FROM market_data
GROUP BY symbol
""")

summary_df = (
    summary_df
    .join(volatility_df, "symbol")
    .join(activity_df, "symbol")
)

summary_df.show()

+--------+-------------+--------------------+------------+----------------------+--------------------+--------+----------+----------+--------------------+--------------------+--------------------+--------------------+---------------+-------------+
|  symbol|total_records|      average_volume|total_trades|average_percent_change| average_price_range|up_count|down_count|flat_count|     avg_price_range|     min_price_range|     max_price_range|  stddev_price_range|volatility_rank|activity_rank|
+--------+-------------+--------------------+------------+----------------------+--------------------+--------+----------+----------+--------------------+--------------------+--------------------+--------------------+---------------+-------------+
| BTCUSDT|          837|   817.5971326164885|1.35409828E8|  -0.01687054050374...|    449.005149342891|     399|       437|         1|    449.005149342891|   59.06999999999971|  3254.1699999999983|   320.7879611347329|              1|            1|
| ETHUSD

In [93]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW volatility_rankings AS

SELECT
    symbol,
    AVG(price_range) AS avg_price_range,
    MIN(price_range) AS min_price_range,
    MAX(price_range) AS max_price_range,
    STDDEV(price_range) AS stddev_price_range,

    DENSE_RANK() OVER(
        ORDER BY STDDEV(price_range) DESC
    ) AS volatility_rank

FROM market_data
GROUP BY symbol
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW activity_rankings AS

SELECT
    symbol,
    SUM(trade_count) AS total_trades_rank,
    SUM(quote_volume) AS total_quote_volume,

    DENSE_RANK() OVER(
        ORDER BY SUM(trade_count) DESC,
                 SUM(quote_volume) DESC
    ) AS activity_rank

FROM market_data
GROUP BY symbol
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW symbol_summary AS

SELECT
    symbol,

    COUNT(*) AS total_records,

    AVG(volume) AS average_volume,

    SUM(trade_count) AS total_trades,

    AVG(percent_change) AS average_percent_change,

    AVG(price_range) AS average_price_range,


    SUM(
        CASE
            WHEN candle_direction = 'up' THEN 1
            ELSE 0
        END
    ) AS up_count,


    SUM(
        CASE
            WHEN candle_direction = 'down' THEN 1
            ELSE 0
        END
    ) AS down_count,


    SUM(
        CASE
            WHEN candle_direction = 'flat' THEN 1
            ELSE 0
        END
    ) AS flat_count


FROM market_data
GROUP BY symbol
""")

final_summary = spark.sql("""
SELECT

    s.symbol,

    s.total_records,
    s.average_volume,
    s.total_trades,
    s.average_percent_change,
    s.average_price_range,

    s.up_count,
    s.down_count,
    s.flat_count,

    v.avg_price_range,
    v.min_price_range,
    v.max_price_range,
    v.stddev_price_range,
    v.volatility_rank,

    a.total_quote_volume,
    a.activity_rank,


    CASE

        WHEN a.activity_rank = 1
            AND v.volatility_rank = 1
            THEN 'Highest activity and highest volatility'

        WHEN a.activity_rank = 1
            THEN 'Highest trading activity'

        WHEN v.volatility_rank = 1
            THEN 'Highest volatility'

        ELSE 'Normal market activity'

    END AS interpretation


FROM symbol_summary s

JOIN volatility_rankings v
ON s.symbol = v.symbol

JOIN activity_rankings a
ON s.symbol = a.symbol


ORDER BY activity_rank
""")

final_summary.show()



+--------+-------------+--------------------+------------+----------------------+--------------------+--------+----------+----------+--------------------+--------------------+--------------------+--------------------+---------------+--------------------+-------------+--------------------+
|  symbol|total_records|      average_volume|total_trades|average_percent_change| average_price_range|up_count|down_count|flat_count|     avg_price_range|     min_price_range|     max_price_range|  stddev_price_range|volatility_rank|  total_quote_volume|activity_rank|      interpretation|
+--------+-------------+--------------------+------------+----------------------+--------------------+--------+----------+----------+--------------------+--------------------+--------------------+--------------------+---------------+--------------------+-------------+--------------------+
| BTCUSDT|          837|   817.5971326164885|1.35409828E8|  -0.01687054050374...|    449.005149342891|     399|       437|        

In [112]:
from shutil import rmtree
rmtree("results/spark_market_summary.csv")
#rmtree("spark-warehouse")

output_path = os.path.join(ROOT_DIR, "results/spark_market_summary")

final_summary.coalesce(1).write \
    .option("header", True) \
    .mode("overwrite") \
    .csv(output_path)


"Spark writes CSV output as a directory because each Spark partition produces its own output file."
"The final summary was reduced to one partition using coalesce(1) because the report contains only one row per symbol."


In [113]:
files.download('/content/results/spark_market_summary/part-00000-6414ae33-a784-4b2c-9f02-349078c6cc83-c000.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [114]:
rows = final_summary.count()

top_activity_symbols = [
    row.symbol
    for row in final_summary.filter(col("activity_rank") == 1).select("symbol").collect()
]

top_volatility_symbols = [
    row.symbol
    for row in final_summary.filter(col("volatility_rank") == 1).select("symbol").collect()
]

print("Final ranked market summary created")
print(f"Rows in summary: {rows}")
print(f"Saved: results/spark_market_summary.csv")
print()
print(f"Top activity symbol(s): {', '.join(top_activity_symbols)}")
print(f"Top volatility symbol(s): {', '.join(top_volatility_symbols)}")
print(
    f"Interpretation: {top_activity_symbols} had the highest trading activity, "
    f"while {top_volatility_symbols} had the highest volatility based on price range."
)

Final ranked market summary created
Rows in summary: 10
Saved: results/spark_market_summary.csv

Top activity symbol(s): BTCUSDT
Top volatility symbol(s): BTCUSDT
Interpretation: ['BTCUSDT'] had the highest trading activity, while ['BTCUSDT'] had the highest volatility based on price range.


In [115]:
spark.stop()